<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@5a33b79/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/pandas_crypto/cours/seance2_cours.ipynb)

# Séance 4 : nettoyer et regrouper

**Cours** · 2h · pandas, seconde partie

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- repérer les quatre défauts d'un fichier réel, et les réparer un par un
- faire sur une colonne de texte ce que vous faisiez sur une chaîne
- compter les lignes par catégorie
- transformer une colonne de texte en dates, et en tirer l'année ou le mois
- répondre à « combien par ... ? » avec `groupby`
- tracer une courbe et des barres

## Les deux phrases du bloc

> **Tout ce que vous savez faire sur une valeur, vous le faites maintenant sur une colonne entière.**

> **Tout ce que pandas fait, vous pourriez l'écrire vous-même avec une colonne de booléens et une boucle.**

La seconde organise cette séance : à chaque fois, on répare d'abord avec ce qu'on sait, puis on voit la fonction pandas qui fait pareil.

Exécutez d'abord la cellule de setup, puis la cellule `jouet`.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 12)      # affichage court sur petit écran
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@main/pandas_crypto/data/"


def verifier(nom, condition, indice=""):
    if condition:
        print("OK       -", nom)
    else:
        print("A REVOIR -", nom, ":", indice)

In [ ]:
# La table jouet de la séance 3, avec une colonne date en plus
coin   = pd.Series(["BTC", "ETH", "SOL", "BTC", "ETH", "SOL"])
date   = pd.Series(["2024-01-01", "2024-01-01", "2024-01-01", "2024-01-02", "2024-01-02", "2024-01-02"])
close  = pd.Series([42000, 2300, 100, 44000, 2400, 95])
volume = pd.Series([20, 10, 5, 25, 12, 4])

jouet = pd.DataFrame({"coin": coin, "date": date, "close": close, "volume": volume})
jouet

In [ ]:
crypto = pd.read_csv(BASE + "crypto.csv")
crypto.shape

## 0. Échauffement

Sur la séance 3.

**1.** Le cours moyen de BNB, dans `moy_bnb`.

In [ ]:
verifier("moy_bnb", abs(moy_bnb - crypto.query("coin == 'BNB'")["close"].mean()) < 0.01, "query, colonne, .mean()")

**2.** Le nombre de jours où DOGE a dépassé 0,50 dollar, dans `nb_doge`.

In [ ]:
verifier("nb_doge", nb_doge == len(crypto.query("coin == 'DOGE' and close > 0.5")), "len d'un query à deux conditions")

**3.** Les trois jours de plus haut cours de SOL, dans `top_sol`.

In [ ]:
verifier("top_sol", len(top_sol) == 3 and (top_sol["coin"] == "SOL").sum() == 3 and top_sol["close"].iloc[0] >= top_sol["close"].iloc[2], "query, sort_values(ascending=False), head(3)")

**4.** Cette cellule est fausse. Exécutez, lisez la dernière ligne, réparez.

In [ ]:
crypto.query("coin == 'ETH' and close > 4000).shape

## 1. Le fichier sale

`crypto.csv` était propre. Un export réel ne l'est jamais. Voici le même fichier pour l'année 2024, tel qu'il sortirait d'un système mal réglé.

In [ ]:
sale = pd.read_csv(BASE + "crypto_sale.csv")
sale.head(8)

In [ ]:
sale.info()

### Le diagnostic

À partir de ces deux cellules, ensemble :

| Ce qu'on voit | Le défaut |
|---|---|
| `close` est de type `object`, et `head` montre `44 167,33 $` | des **nombres écrits en texte** |
| `volume` a moins de `non-null` que de lignes | des **valeurs manquantes** |
| `coin` contient `btc`, ` BTC`, `Btc ` | des **catégories incohérentes** |
| `info()` ne le dit pas, mais il faut vérifier | des **lignes en double** |

Quatre défauts, quatre sections. Le plan de la séance est écrit par le diagnostic.

Et **la seconde phrase du bloc** : pour chaque défaut, on répare d'abord avec ce qu'on sait, puis on voit la fonction pandas qui fait pareil.

## 2. Les valeurs manquantes

`isna()` pose une question à chaque case : « es-tu vide ? ». Sur une colonne, ça donne une Series de `bool`, comme une comparaison.

In [ ]:
sale["volume"].isna()

Compter, comme à la séance 3 :

In [ ]:
sale["volume"].isna().sum()

Sur la table entière, une réponse par colonne :

In [ ]:
sale.isna().sum()

### Réparez-le vous-même

On veut garder les lignes où le volume n'est **pas** manquant. On a une Series de `bool`. On en fait une **colonne**, et `query` sait filtrer sur une colonne.

In [ ]:
sale["manquant"] = sale["volume"].isna()
sans_trous = sale.query("manquant == False")
len(sale), len(sans_trous)

### Un `if` sur chaque ligne

Une colonne de `bool`, c'est la liste des réponses qu'un `if` donnerait ligne par ligne. Et un `if` accepte un booléen tout seul :

In [ ]:
if True:
    print("toujours")
if False:
    print("jamais")

`query("manquant == False")`, c'est ce `if` fait sur chaque ligne d'un coup : on garde la ligne si sa case vaut `False`.

### La fonction pandas

`dropna` fait exactement ce que vous venez d'écrire, sans colonne intermédiaire.

In [ ]:
propre = sale.dropna(subset=["volume"])
len(propre)

`subset` dit sur quelle colonne regarder. Sans lui, une ligne avec un trou n'importe où serait retirée.

On note le nombre de lignes **avant** et **après**. On le fera à chaque étape : c'est ce qu'on rapporte à la fin.

### Prédire

`jouet2` est `jouet` avec deux volumes manquants.

In [ ]:
jouet2 = jouet.copy()
jouet2["volume"] = pd.Series([20, np.nan, 5, 25, np.nan, 4])
jouet2

In [ ]:
# Prédiction :
jouet2["volume"].isna()

In [ ]:
# Prédiction :
jouet2["volume"].isna().sum()

In [ ]:
# Prédiction :
len(jouet2.dropna(subset=["volume"]))

In [ ]:
# Prédiction :
jouet2.dropna(subset=["volume"])["volume"].sum()

### Écrire

Sur `sale` : le nombre de lignes avec un volume manquant, dans `nb_trous`. Puis `net`, la table sans ces lignes, par la méthode de votre choix.

In [ ]:
verifier("nb_trous", nb_trous == 41, "isna() puis .sum()")
verifier("net", len(net) == len(sale) - nb_trous, "dropna(subset=[...]) ou query sur une colonne de bool")

## 3. Les doublons

`duplicated()` pose une question à chaque ligne : « t'ai-je déjà vue plus haut ? ». Une Series de `bool`, encore.

In [ ]:
net.duplicated().sum()

### Réparez-le vous-même

In [ ]:
net["doublon"] = net.duplicated()
sans_doublons = net.query("doublon == False")
len(net), len(sans_doublons)

### La fonction pandas

Avant, on retire les deux colonnes de travail, `manquant` et `doublon` : on garde les quatre colonnes de départ. Sinon, une ligne et sa copie ne seraient plus identiques (`False` d'un côté, `True` de l'autre), et `drop_duplicates` ne verrait plus de doublon.

In [ ]:
net = net[["date", "coin", "close", "volume"]]
net = net.drop_duplicates()
len(net)

### Lire le code, dire la question

À quelle question répond cette ligne ?

In [ ]:
sans_doublons.query("doublon == True")

Une table vide : dans `sans_doublons`, plus aucune ligne n'est un doublon. Et sur `net` avant nettoyage, la même question donnait les 30 lignes en double, à regarder avant de supprimer.

### Écrire

`sale2` est une copie de `sale`. Enlevez les doublons **puis** les lignes sans volume, dans `net2`, et comparez avec `net`.

In [ ]:
sale2 = sale.copy()

In [ ]:
verifier("net2", len(net2) == 2522 and len(net2) == len(net), "l'ordre des deux opérations ne change rien ici")

## 4. Des nombres écrits en texte

`close` est du texte. On ne peut rien calculer avec.

In [ ]:
net["close"].max()

Ça renvoie un texte, et le « maximum » de textes ne veut rien dire.

En séance 1, on nettoyait **un** texte avec `replace` puis `float` :

In [ ]:
float("43 250,12 $".replace(" ", "").replace(",", ".").replace("$", ""))

Sur une colonne, c'est la même chose, avec une différence de syntaxe.

### Pourquoi `.str`

`net["close"]` est une Series, pas un texte. Pour dire « applique la méthode de texte **à chaque case** », on écrit `.str` devant la méthode :

```python
net["close"].str.replace(" ", "")
```

`.str` veut dire : traite chaque case comme une chaîne.

In [ ]:
texte = net["close"].str.replace(" ", "").str.replace(",", ".").str.replace("$", "")
texte.head(3)

Puis la conversion : `astype(float)`, la version colonne de `float()`.

In [ ]:
net["close"] = texte.astype(float)
net.info()

`close` est passé en `float64`. Le maximum a maintenant un sens.

In [ ]:
net["close"].max()

### Prédire

`jouet3` est `jouet` avec des prix en texte.

In [ ]:
jouet3 = jouet.copy()
jouet3["close"] = pd.Series(["42 000,00", "2 300,00", "100,00", "44 000,00", "2 400,00", "95,00"])
jouet3

In [ ]:
# Prédiction :
jouet3["close"].str.replace(" ", "")

In [ ]:
# Prédiction :
jouet3["close"].str.replace(" ", "").str.replace(",", ".").astype(float).sum()

### Corriger

In [ ]:
# Pas d'erreur, et pourtant rien ne change : comparez avec la cellule suivante
jouet3["close"].replace(" ", "")

In [ ]:
jouet3["close"].str.replace(" ", "")

In [ ]:
# Une erreur, et sa dernière ligne cite la valeur fautive
jouet3["close"].astype(float)

## 5. Le texte, c'est des catégories

On vient de transformer du texte en nombres, parce que c'était des nombres mal écrits. Mais le plus souvent, le texte d'une table n'est pas un nombre déguisé.

Dans une table d'analyse, une colonne de texte, c'est de deux choses l'une :

- une **date** : on s'en occupe à la section suivante ;
- une **catégorie** : la monnaie, le pays, le secteur, le type de client.

Ici, `coin` est une catégorie. Et la première question qu'on pose à une catégorie, c'est : **combien de lignes par catégorie ?** C'est `value_counts`.

In [ ]:
crypto["coin"].value_counts()

Une Series : l'index, ce sont les catégories ; les valeurs, les effectifs, du plus fréquent au moins fréquent. On y lit que SOL a moins de jours : elle est apparue plus tard.

Les catégories elles-mêmes, et leur nombre :

In [ ]:
crypto["coin"].unique()

In [ ]:
crypto["coin"].nunique()

### Les catégories incohérentes

La même chose sur le fichier sale :

In [ ]:
net["coin"].value_counts()

Vingt-huit catégories pour sept monnaies. Pour pandas, `"btc"` et `" BTC"` sont deux catégories, et chacune est sous-comptée.

On répare avec `strip` et `upper`, en `.str`, comme en séance 1.

In [ ]:
net["coin"] = net["coin"].str.strip().str.upper()
net["coin"].value_counts()

Sept catégories. Le nettoyage d'une catégorie, c'est presque toujours ces deux méthodes.

### Prédire

In [ ]:
# Prédiction :
jouet["coin"].value_counts()

In [ ]:
# Prédiction :
jouet["coin"].nunique()

In [ ]:
# Prédiction :
jouet["date"].value_counts()

In [ ]:
# Prédiction :
jouet["coin"].str.lower().unique()

### Écrire

**1.** Combien de catégories `coin` avait-il dans `sale`, avant nettoyage ? Dans `nb_avant`.

In [ ]:
verifier("nb_avant", nb_avant == 28, "nunique() sur la colonne coin de sale")

**2.** Le volume moyen de BTC dans `net`, dans `vol_btc`. Un `query` sur `coin`, qui ne marche que si le nettoyage est fait.

In [ ]:
verifier("vol_btc", abs(vol_btc - net.query("coin == 'BTC'")["volume"].mean()) < 1, "query sur coin == 'BTC', puis .mean() du volume")

## 6. Les dates

`date` est du texte. On peut la comparer, on l'a fait à la séance 3, mais on ne peut pas lui demander « quel mois ? ».

`pd.to_datetime` la convertit en vraie date.

In [ ]:
net["date"] = pd.to_datetime(net["date"])
net.info()

Une fois convertie, `.dt` donne accès aux morceaux, comme `.str` donnait accès aux méthodes de texte.

In [ ]:
net["annee"] = net["date"].dt.year
net["mois"] = net["date"].dt.month
net.head(3)

`.dt.day` existe aussi : le jour du mois.

Même chose sur `crypto`, parce que la suite travaille dessus :

In [ ]:
crypto["date"] = pd.to_datetime(crypto["date"])
crypto["annee"] = crypto["date"].dt.year
crypto.head(3)

### Écrire

Le nombre de lignes de `crypto` en 2021, dans `nb_2021`, avec la colonne `annee`. Puis le cours maximum de BTC en 2021, dans `btc_max_2021`.

In [ ]:
verifier("nb_2021", nb_2021 == 2555, "query sur annee == 2021, puis len")
verifier("btc_max_2021", abs(btc_max_2021 - 67566.8281) < 0.01, "deux conditions, puis .max()")

## 7. Regrouper : `groupby`

La section la plus importante du bloc. On y va pas à pas, sur `jouet` d'abord.

### La question

« Le cours moyen de chaque monnaie. » Trois monnaies dans `jouet`, trois moyennes. Avec ce qu'on sait, on écrit un `query` par monnaie :

In [ ]:
jouet.query("coin == 'BTC'")["close"].mean()

Trois fois. Sept fois sur le vrai fichier. Non : **une boucle**.

La condition de `query` est un texte, donc on la fabrique avec une f-string, comme en séance 2.

In [ ]:
monnaies = ["BTC", "ETH", "SOL"]
moyennes = []
for m in monnaies:
    sous_table = jouet.query(f"coin == '{m}'")
    moyennes.append(sous_table["close"].mean())
pd.Series(moyennes, index=monnaies)

### Ce que fait la boucle

```
       jouet                    découper                  calculer            rassembler
                               (une sous-table
   coin   close                 par monnaie)             (une moyenne         (une Series,
   BTC   42000                                            par sous-table)      une ligne
   ETH    2300          BTC │ 42000 │ 44000  ──►  43000                       par monnaie)
   SOL     100    ──►   ETH │  2300 │  2400  ──►   2350       ──►     BTC   43000.0
   BTC   44000          SOL │   100 │    95  ──►     97.5              ETH    2350.0
   ETH    2400                                                         SOL      97.5
   SOL      95
```

Trois temps :

1. **découper** la table en une sous-table par valeur de la colonne choisie ;
2. **calculer** une chose dans chaque sous-table ;
3. **rassembler** les résultats dans une Series dont l'index est la catégorie.

### La ligne pandas

`groupby` fait ces trois temps en une ligne.

In [ ]:
jouet.groupby("coin")["close"].mean()

Même résultat, à la ligne près. Lecture de gauche à droite :

```
jouet.groupby("coin")["close"].mean()
│     │              │        │
│     │              │        └─ calculer : la moyenne dans chaque sous-table
│     │              └─ de la colonne close
│     └─ découper : une sous-table par valeur de coin
└─ la table
```

Le résultat est une **Series** : une valeur par groupe, l'index est le groupe. Tout ce qu'on sait faire sur une Series s'applique.

### Sur le vrai fichier

La même ligne.

In [ ]:
crypto.groupby("coin")["close"].mean()

### Ce qu'on met à la fin

N'importe quelle fonction de colonne.

In [ ]:
crypto.groupby("coin")["close"].max()

In [ ]:
crypto.groupby("coin")["volume"].sum()

In [ ]:
crypto.groupby("coin")["close"].count()

`count` compte les lignes de chaque groupe : c'est `value_counts`, obtenu autrement.

### Regrouper par autre chose

Par année, avec la colonne créée à la section 6. Une catégorie, c'est n'importe quelle colonne qui prend peu de valeurs différentes.

In [ ]:
crypto.groupby("annee")["volume"].sum()

### Trier le résultat, aller chercher un groupe

In [ ]:
crypto.groupby("coin")["close"].mean().sort_values(ascending=False)

In [ ]:
crypto.groupby("coin")["close"].mean()["ETH"]

### Prédire

In [ ]:
# Prédiction :
jouet.groupby("coin")["volume"].sum()

In [ ]:
# Prédiction :
jouet.groupby("date")["volume"].sum()

In [ ]:
# Prédiction :
jouet.groupby("coin")["close"].count()

In [ ]:
# Prédiction :
jouet.groupby("coin")["close"].max().sort_values()

In [ ]:
# Prédiction :
jouet.groupby("coin")["close"].mean()["ETH"]

In [ ]:
# Prédiction :
jouet.groupby("coin")["close"].max() - jouet.groupby("coin")["close"].min()

La dernière : deux Series avec le même index se soustraient étiquette par étiquette. L'amplitude de chaque monnaie, en une ligne.

### Lire le code, dire la question

Pour chaque ligne, à l'oral : à quelle question répond-elle ?

In [ ]:
crypto.groupby("annee")["close"].max()

In [ ]:
crypto.query("coin == 'BTC'").groupby("annee")["close"].mean()

In [ ]:
crypto.groupby("coin")["annee"].min()

### Dire la question, écrire le code

Au tableau, ensemble :

- le volume moyen par monnaie, du plus gros au plus petit ;
- le cours minimum de chaque année pour ETH.

### Écrire

**1.** Le cours moyen de chaque monnaie en 2024, dans `moy_2024`. Un `query`, puis un `groupby`. Quelle monnaie est en tête ?

In [ ]:
verifier("moy_2024", type(moy_2024) == pd.Series and len(moy_2024) == 7 and abs(moy_2024["BTC"] - 65964.12) < 0.01, "query sur annee == 2024, puis groupby('coin')['close'].mean()")

**2.** Le nombre de jours dans le fichier pour chaque monnaie, dans `nb_jours`, trié. Comparez avec `value_counts`.

In [ ]:
verifier("nb_jours", len(nb_jours) == 7 and nb_jours["SOL"] == 2335 and nb_jours["BTC"] == 3165, "groupby('coin')['close'].count(), puis sort_values()")

**3.** Refaites `moy_2024` avec la boucle et les `query`, sans `groupby`, dans `moy_2024_boucle`. Les deux Series doivent avoir les mêmes valeurs. Le but est de sentir que c'est la même chose.

In [ ]:
verifier("moy_2024_boucle", len(moy_2024_boucle) == 7 and all(abs(moy_2024_boucle[m] - moy_2024[m]) < 0.01 for m in moy_2024.index), "une liste vide, une boucle sur les monnaies, un query avec f-string, un append, une Series")

### Corriger

La première cellule ne produit pas d'erreur. Regardez ce qu'elle renvoie, et dites pourquoi ce n'est pas ce qu'on voulait.

In [ ]:
crypto.groupby("coin").mean()

In [ ]:
crypto.groupby(annee)["close"].mean()

## 8. Tracer

Deux graphiques, deux lignes.

Une Series de `groupby` se trace en **barres** :

In [ ]:
crypto.groupby("coin")["close"].mean().sort_values().plot(kind="bar")

Une évolution dans le temps se trace en **courbe**, en disant quelle colonne va en abscisse :

In [ ]:
crypto.query("coin == 'BTC'").plot(x="date", y="close")

### Écrire

La courbe de ETH depuis 2023. Un `query` à deux conditions, puis `plot`.

## 9. Ce que vous savez faire

| Vous voulez... | Vous écrivez |
|---|---|
| repérer les manquants | `df["col"].isna()`, `df.isna().sum()` |
| retirer les lignes incomplètes | `df.dropna(subset=["col"])` |
| repérer les doublons | `df.duplicated().sum()` |
| retirer les doublons | `df.drop_duplicates()` |
| une méthode de texte sur une colonne | `df["col"].str.replace(",", ".")`, `.str.strip()`, `.str.upper()` |
| texte vers nombre | `df["col"].astype(float)` |
| compter par catégorie | `df["col"].value_counts()` |
| les catégories, leur nombre | `df["col"].unique()`, `df["col"].nunique()` |
| texte vers date | `df["date"] = pd.to_datetime(df["date"])` |
| l'année, le mois, le jour | `df["date"].dt.year`, `.dt.month`, `.dt.day` |
| combien par catégorie, d'une autre colonne | `df.groupby("coin")["close"].mean()` |
| trier le résultat | `.sort_values(ascending=False)` |
| un groupe | `df.groupby("coin")["close"].mean()["ETH"]` |
| des barres | `serie.plot(kind="bar")` |
| une courbe | `df.plot(x="date", y="close")` |

## Les deux phrases du bloc

1. **Tout ce que vous savez faire sur une valeur, vous le faites maintenant sur une colonne entière.** `.str` et `.dt` sont la façon de le dire pour le texte et les dates.
2. **Tout ce que pandas fait, vous pourriez l'écrire vous-même avec une colonne de booléens et une boucle.** `dropna`, `drop_duplicates` et `groupby` : vous les avez écrits avant de les apprendre.

## Ce que vous croiserez ailleurs

Ces outils existent, vous les verrez dans des exemples en ligne. Une ligne chacun, pour les reconnaître :

| Vous verrez | Ce que ça fait |
|---|---|
| `df[df["close"] > 100]` | un `query` écrit autrement, avec `&` pour `and` et `\|` pour `or` |
| `df.loc[3, "close"]`, `df.iloc[0]` | une case ou une ligne par son étiquette ou sa position |
| `df.merge(autre, on="col")` | coller deux tables qui partagent une colonne |
| `df.describe()` | huit statistiques d'un coup : c'est le début du bloc suivant |

## La suite

L'exercice de synthèse : si vous aviez acheté 100 dollars de bitcoin le premier de chaque mois depuis 2020, combien auriez-vous aujourd'hui ? Tout ce qu'il demande est dans ces deux séances.